# HDI Cross-Country Analysis (2022): A Multi-Source Data Pipeline & Structural Audit
## Notebook 01: Data acquisition
### Goal: Pull all raw datasets programmatically using APIs and save them to `/raw_data`

## Data Sources & Licenses

| Dataset | Source | License | URL |
|---|---|---|---|
| GDP per capita, Education & Health Expenditure | World Bank | CC BY 4.0 | https://data.worldbank.org |
| Country Metadata (Region & Income Level) | World Bank | CC BY 4.0 | https://data.worldbank.org |
| Human Development Index, Mean Years of Schooling | UNDP | CC BY 3.0 IGO | https://hdr.undp.org |
| Gender Inequality Index | UNDP | CC BY 3.0 IGO | https://hdr.undp.org |
| Government Effectiveness Index | World Bank (WGI) | CC BY 4.0 | https://info.worldbank.org/governance/wgi |

**CC BY 4.0**: Free to use, share, and adapt with attribution  
**CC BY 3.0 IGO**: Same as above but specifically for intergovernmental organizations

In [1]:
# Data Acquisition Setup

import requests        
from io import BytesIO  
import wbgapi as wb    
import pandas as pd    
from pathlib import Path
import os

BASE_DIR = Path("..").resolve()
RAW_DATA = BASE_DIR / "raw_data"
RAW_DATA.mkdir(exist_ok=True)

# Mimic browser request to avoid potential blocking by the server
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

print("Libraries loaded")
print(f"Raw data folder: {RAW_DATA}")

Libraries loaded
Raw data folder: E:\Portfolio\projects\HDI_Analysis\raw_data


##  I World Bank Data Pull
Indicators:
- `NY.GDP.PCAP.KD`: GDP per capita (constant 2015 USD)
- `SE.XPD.TOTL.GD.ZS`: Education expenditure (% of GDP)
- `SH.XPD.CHEX.PC.CD`: Health expenditure per capita (current USD)

In [2]:
# World Bank Indicators
indicators = {
    'NY.GDP.PCAP.KD':       'gdp_per_capita',
    'SE.XPD.TOTL.GD.ZS':   'education_expenditure_pct_gdp',
    'SH.XPD.CHEX.PC.CD':   'health_expenditure_per_capita'
}

# Pull 2019–2025 to identify the most recent year with complete coverage
wb_data = wb.data.DataFrame(
    list(indicators.keys()),
    time=range(2019, 2026), # Pull 2019–2025 to identify the most recent year with complete coverage
    labels=True
)
# Save World Bank raw data
wb_data.to_csv(RAW_DATA / "worldbank_raw.csv")
print(f"World Bank raw data saved: Shape: {wb_data.shape}")


World Bank raw data saved: Shape: (798, 9)


## II World Bank Country Metadata
Region and income level classifications sourced directly from the 
World Bank API to enable geographic and economic grouping in analysis 
and visualization.

**Indicators:**
- `region`: World Bank regional grouping (7 regions)
- `incomeLevel`: World Bank income classification (4 tiers)


In [3]:
# World Bank Country Metadata — Region & Income Level 
countries_meta = wb.economy.DataFrame()

# Filter out regional aggregates
countries_meta = countries_meta[countries_meta['aggregate'] == False][['name', 'region', 'incomeLevel']].reset_index()
countries_meta.columns = ['country_code', 'country_name', 'wb_region', 'wb_income_level']

# Map codes to full names
region_map = {
    'EAS': 'East Asia & Pacific',
    'ECS': 'Europe & Central Asia',
    'LCN': 'Latin America & Caribbean',
    'MEA': 'Middle East & North Africa',
    'NAC': 'North America',
    'SAS': 'South Asia',
    'SSF': 'Sub-Saharan Africa'
}

income_map = {
    'HIC': 'High income',
    'UMC': 'Upper middle income',
    'LMC': 'Lower middle income',
    'LIC': 'Low income'
}

countries_meta['region'] = countries_meta['wb_region'].map(region_map)
countries_meta['income_level'] = countries_meta['wb_income_level'].map(income_map)

# Save to raw_data
countries_meta[['country_code', 'region', 'income_level']].to_csv(
    RAW_DATA / "wb_country_metadata_raw.csv", index=False)

print(f"Country metadata saved: {len(countries_meta)} countries")
print(countries_meta[['country_code', 'region', 'income_level']].head())

Country metadata saved: 217 countries
  country_code                      region         income_level
0          ABW   Latin America & Caribbean          High income
1          AFG  Middle East & North Africa           Low income
2          AGO          Sub-Saharan Africa  Lower middle income
3          ALB       Europe & Central Asia  Upper middle income
4          AND       Europe & Central Asia          High income


## III UNDP Human Development Index (HDI)
HDI and mean years of schooling sourced from the UNDP Human Development
Report 2023-24 Statistical Annex.

**Indicators:**
- `HDI`: Human Development Index (2022)
- `Mean years of schooling`: Average years of schooling for adults 25+


In [4]:
# UNDP HDI Data 
undp_url = "https://hdr.undp.org/sites/default/files/2023-24_HDR/HDR23-24_Statistical_Annex_HDI_Table.xlsx"

response = requests.get(undp_url, headers=headers)
print(f"Status code: {response.status_code}")

undp_raw = pd.read_excel(BytesIO(response.content), skiprows=4)
print(f"Shape: {undp_raw.shape}")

Status code: 200
Shape: (272, 15)


In [5]:
# Save UNDP raw data
undp_raw.to_csv(RAW_DATA / "undp_hdi_raw.csv", index=False) 
print(f"UNDP raw data saved")

UNDP raw data saved


## IV World Governance Indicators (WGI)
Source: World Bank
Indicator:
- Government Effectiveness measures quality of public services and policy implementation

In [6]:
# Pull World Governance Indicators
wgi_url = "https://api.worldbank.org/v2/en/indicator/GE.EST?downloadformat=csv"

response = requests.get(wgi_url, headers=headers)
print(f"Status code: {response.status_code}")

# Write binary content to file
wgi_zip_path = RAW_DATA / "wgi_government_effectiveness_raw.zip"
with open(wgi_zip_path, "wb") as f:
    f.write(response.content)

print(f"WGI raw data saved: {wgi_zip_path.name}")

Status code: 200
WGI raw data saved: wgi_government_effectiveness_raw.zip


## V UNDP Gender Inequality Index (GII)
Source: UNDP Human Development Reports
Indicator: Gender Inequality Index (GII) measures gender-based disadvantage

In [7]:
# Pull GII data
gii_url = "https://hdr.undp.org/sites/default/files/2023-24_HDR/HDR23-24_Statistical_Annex_GII_Table.xlsx"

response = requests.get(gii_url, headers=headers)
print(f"Status code: {response.status_code}")

gii_raw = pd.read_excel(BytesIO(response.content), skiprows=4)

# Save raw file 
gii_raw.to_csv(RAW_DATA / "undp_gii_raw.csv", index=False)
print(f"GII raw data saved Shape: {gii_raw.shape}")


Status code: 200
GII raw data saved Shape: (261, 20)


## VI Data verification (all raw files are saved)


In [8]:
# Verify all raw files saved 
files = list(RAW_DATA.iterdir())

print("Files in raw_data folder:")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {f.name}: {size_kb:.1f} KB")

Files in raw_data folder:
  undp_gii_raw.csv: 25.5 KB
  undp_hdi_raw.csv: 22.7 KB
  wb_country_metadata_raw.csv: 9.0 KB
  wgi_government_effectiveness_raw.zip: 58.8 KB
  worldbank_raw.csv: 125.2 KB
